# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Softwareguy786/ML_INTERNSHIP_01/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [1]:
import pandas as pd

# Load the data
df = pd.read_csv('/content/content_refresh_anonymized.csv')

# Select only the columns needed for content refresh prioritization
lane_cols = [
    'content_id',
    'content_type',
    'main_intent',
    'impressions_90d',
    'clicks_90d',
    'sessions_90d',
    'ctr',
    'avg_position',
    'trend_direction',
    'trend_pct',
    'content_age_days',
    'days_since_last_update',
    'freshness_tier',
    'impressions_last_30d',
    'impressions_prev_30d'
]

lane_df = df[lane_cols].copy()

# Quick check: key signals are numeric and spread out enough for scoring
lane_df[['trend_pct', 'ctr', 'avg_position', 'content_age_days', 'days_since_last_update']].describe()

,trend_pct,ctr,avg_position,content_age_days,days_since_last_update
count,26612.000000,30000.000000,30000.00000,30000.00000,30000.000000
mean,-4.785969,0.510733,16.34238,256.16780,46.098300
std,473.861780,3.279162,15.21679,132.70793,42.078709
min,-100.000000,0.000000,0.00000,90.00000,1.000000
25%,-62.600000,0.000000,6.20000,132.00000,20.000000
50%,-33.500000,0.070000,10.80000,236.00000,20.000000
75%,0.000000,0.290000,22.30000,333.00000,104.000000
max,44900.000000,100.000000,245.00000,564.00000,373.000000


## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*



> My lane is content refresh prioritization.
I am framing it as a scoring problem, not classification or clustering.
The model will produce a refresh urgency score for each content page.
Then pages can be ranked from highest to lowest priority.
This is better than classification because an editor needs a ranked queue, not a simple yes/no label.



In [2]:
# Check that key signals are numeric and spread out enough for scoring
lane_df[['trend_pct', 'ctr', 'avg_position', 'content_age_days', 'days_since_last_update']].describe()

,trend_pct,ctr,avg_position,content_age_days,days_since_last_update
count,26612.000000,30000.000000,30000.00000,30000.00000,30000.000000
mean,-4.785969,0.510733,16.34238,256.16780,46.098300
std,473.861780,3.279162,15.21679,132.70793,42.078709
min,-100.000000,0.000000,0.00000,90.00000,1.000000
25%,-62.600000,0.000000,6.20000,132.00000,20.000000
50%,-33.500000,0.070000,10.80000,236.00000,20.000000
75%,0.000000,0.290000,22.30000,333.00000,104.000000
max,44900.000000,100.000000,245.00000,564.00000,373.000000


## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*



> I would predict a refresh urgency score for each content page. There is no observed target in the data, so I will use a defined rule as a proxy. The proxy combines traffic trend (trend_pct), page age (content_age_days), days since last update (days_since_last_update), and current traffic level (impressions_90d). The score will be higher for pages that are declining, old, not recently updated, but still have enough traffic to matter.



In [3]:
# Very rough proxy: higher score = more urgent to refresh
lane_df['refresh_urgency_proxy'] = (
    lane_df['trend_pct'].clip(lower=-100, upper=0).abs() / 100 * 0.4 +
    lane_df['content_age_days'] / lane_df['content_age_days'].max() * 0.3 +
    lane_df['days_since_last_update'] / lane_df['days_since_last_update'].max() * 0.2 +
    lane_df['impressions_90d'] / lane_df['impressions_90d'].max() * 0.1
)

lane_df[['content_id', 'trend_pct', 'content_age_days', 'days_since_last_update', 'impressions_90d', 'refresh_urgency_proxy']].sort_values('refresh_urgency_proxy', ascending=False).head(10)

,content_id,trend_pct,content_age_days,days_since_last_update,impressions_90d,refresh_urgency_proxy
29384,content_f6fdf87348f6,-100.0,373,373,2,0.798405
24216,content_1b4ec72dafd4,-100.0,372,372,2,0.797337
26242,content_55a5b1c46474,-88.5,374,373,35,0.752943
8407,content_d1e915d03c28,-100.0,537,104,2,0.741403
7509,content_7a888d3d99c8,-100.0,313,313,95,0.734336
18841,content_94991fe6268c,-100.0,313,313,7,0.734319
22860,content_ab18b5811c02,-100.0,313,305,5,0.730029
24557,content_84d12054c0c0,-100.0,305,304,1,0.725237
19420,content_ccfb4d0227b1,-100.0,301,301,8,0.721502
21984,content_02b0d6e30129,-95.6,313,313,176,0.716752


## 3. Success metric

*One metric you can defend. What number means 'good'?*



> The success metric is: for the top 20 pages flagged by the model, editors refresh them and within 30 days their combined impressions increase by at least 20% compared to the previous 30 days before refresh. This is a measurable business outcome that ties directly to the model’s ranked queue.



In [4]:
# Example: what share of pages are currently losing impressions?
losing_pages = lane_df['trend_pct'] < 0
print(f"Pages with negative trend: {losing_pages.sum()} / {len(lane_df)} ({losing_pages.mean():.1%})")

Pages with negative trend: 19715 / 30000 (65.7%)


## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

import pandas as pd
df = pd.read_csv('/content/content_refresh_anonymized.csv')
df.shape
df.info()
df.head()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 30000 entries, 0 to 29999
Data columns (total 44 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   content_id              30000 non-null  object 
 1   client_id               30000 non-null  object 
 2   search_volume           27532 non-null  float64
 3   competition             27532 non-null  float64
 4   competition_level       27390 non-null  object 
 5   cpc                     27532 non-null  float64
 6   content_type            30000 non-null  object 
 7   main_intent             27626 non-null  object 
 8   word_count              22301 non-null  float64
 9   char_count              22301 non-null  float64
 10  provider_used           8562 non-null   object 
 11  model_used              24267 non-null  object 
 12  impressions_90d         30000 non-null  int64  
 13  clicks_90d              30000 non-null  int64  
 14  pageviews_90d           30000 non-null

,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,char_count_tier,ctr,avg_position,engagement_rate,scroll_rate,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct
0,content_304f48230142,client_f369cb89fc,10.0,0.67,HIGH,2.05,keyword article,transactional,3221.0,20457.0,...,15000-25000,0.76,10.6,5.88,4.55,0.0,good,striking,down,-41.4
1,content_a1fb4e703a9e,client_4e07408562,90.0,0.01,LOW,0.05,keyword article,informational,2481.0,15562.0,...,15000-25000,0.05,20.3,0.00,10.00,0.0,good,page_3_5,down,-57.7
2,content_9aa793d4d895,client_7f2253d7e2,0.0,0.00,LOW,0.00,keyword article,informational,3515.0,23643.0,...,15000-25000,0.09,36.5,0.00,28.57,0.0,good,page_3_5,down,-60.9
3,content_331d6c4de07b,client_19581e27de,10.0,0.00,LOW,0.00,keyword article,commercial,NaN,NaN,...,NaN,0.49,6.2,1.28,3.45,0.0,good,page_1,stable,-13.8
4,content_d99b7a2d90ca,client_3fdba35f04,0.0,0.00,LOW,0.00,keyword article,informational,2803.0,17469.0,...,15000-25000,0.13,44.0,0.00,24.29,0.0,good,page_3_5,down,-34.7




> One row is one content page / content asset, identified by content_id



In [6]:
df['content_id'].is_unique

True



> It means that one row = one content item



In [7]:
df['content_id'].nunique()

30000



> It mean all IDs are different.



**Since we are working on content refresh priortization we don't need all cols.**



> A good slice would include:

    Identifier: content_id

    Performance: impressions_90d, clicks_90d, sessions_90d, ctr, avg_position

    Trend: trend_direction, trend_pct

    Freshness: content_age_days, days_since_last_update, freshness_tier

    Optional: content_type, main_intent, impressions_last_30d, impressions_prev_30d



In [8]:
# lane_cols = [
#     'content_id',
#     'content_type',
#     'main_intent',
#     'impressions_90d',
#     'clicks_90d',
#     'sessions_90d',
#     'ctr',
#     'avg_position',
#     'trend_direction',
#     'trend_pct',
#     'content_age_days',
#     'days_since_last_update',
#     'freshness_tier',
#     'impressions_last_30d',
#     'impressions_prev_30d'
# ]

# lane_df = df[lane_cols].copy()
# lane_df.head()



> The dataframe below is my lane slice: one row per content page, with columns for performance, trend, and freshness that will be used to score or rank refresh priority.



## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*



> A fixed rule like “refresh if content_age_days > 200 and trend_pct < -30” is too simple. It ignores pages that are old but still growing, or newer pages that are dropping fast. It also treats all drops equally, even if a page gets huge traffic and a small drop is more important than a large drop on a near-zero page. ML can combine multiple signals, learn patterns from historical refresh outcomes, and adapt as the data changes. That makes it better than a hard-coded if-statement.



In [9]:
# Show limitation of a simple rule
simple_rule = (lane_df['content_age_days'] > 300) & (lane_df['trend_pct'] < -50)
print(f"Simple rule flags: {simple_rule.sum()} pages")

# But a high-impression page with moderate decline may be missed
missed = lane_df[(lane_df['impressions_90d'] > 5000) & (lane_df['trend_pct'] < -20) & (~simple_rule)]
print(f"High-impression pages missed by simple rule: {len(missed)}")

Simple rule flags: 2295 pages
High-impression pages missed by simple rule: 3092
